> **Run-Notebook `pythia-1.4b-lite`** (erzeugt von `scripts/make_run_notebooks.py`, nicht von Hand bearbeiten): MODEL `pythia-1.4b`, Drive-Ordner `MyDrive/ValueHeadsPythia/pythia-1.4b`, Werte `honesty harmless helpful`, ohne Likelihood-Evaluation und Generation/Judge (Abschnitte 9 und 10 entfernt). Laufzeit: A100 (Colab Pro+), alle Zellen ausführen.

# ValueHeads auf Pythia – Colab-Notebook

Port der ValueHeads-Methode (arXiv 2611.02503, Referenz-Code sihuo-design/valueheads, MIT) auf die Pythia-Suite:
Head-Lokalisierung per linearer Probe, Masken-/Steering-Interventionen, judge-freie Likelihood-Evaluation und
Checkpoint-Sweep. Das Notebook ruft ausschließlich die CLI und die Python-API des Projekts `valueheads_pythia`
auf (siehe `README.md`, verbindliche Spezifikation `DESIGN.md`).

**Ablauf (12 Zellen, DESIGN Abschnitt 7):** 1 Laufzeit · 2 Installation/Projekt/Drive · 3 Konfiguration · 4 Selftest ·
5 Extraktion · 6 Probing + Statistik + Kontrollen · 7 Heatmaps · 8 Richtungen · 9 Likelihood-Evaluation ·
10 Generation + Judge (optional) · 11 Sweep · 12 Analyse + Drive-Sync.

**Vor dem Start:** Laufzeittyp wählen (T4 / L4 / A100 oder CPU für den Smoke-Test). Das Projekt wird per Default aus
GitHub geklont (`PROJECT_SOURCE = "git"`, Zelle 2; alternativ `"drive"` = Ordner `ValueHeadsPythia/valueheads_pythia` auf Google Drive). Ergebnisse (`results/`,
`figures/`) landen auf Drive unter `ValueHeadsPythia/`, Features auf der lokalen Colab-Disk `/content/features`.

**Datenschutz-Hinweis:** HH-RLHF-, BeaverTails- und Jailbreak-Inhalte werden nie ausgegeben; vor dem Speichern des
Notebooks trotzdem immer *Bearbeiten → Alle Ausgaben löschen* ausführen (letzte Zelle).

## 1. Laufzeit-Check

GPU-Typ → `TIER` (`cpu` / `t4` / `l4` / `a100`), RAM/Disk, Versionen, `HF_HOME` auf der lokalen Disk
(`/content/hf_cache`, nicht auf Drive: FUSE ist für große Safetensors unzuverlässig). `SMOKE=True` schaltet auf die
kleine Konfiguration (winzige Teilmengen, nur CPU-taugliche Zellen); auf einer CPU-Laufzeit ist das automatisch der
Fall.

In [ ]:
# 1. Laufzeit-Check: GPU-Typ -> TIER, RAM / Disk, Versionen, HF_HOME
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# SMOKE: tiny subsets, CPU-compatible cells only (env VH_NB_SMOKE=1, or set it by hand; CPU runtimes always use it)
SMOKE = os.environ.get("VH_NB_SMOKE", "0") == "1"

import torch

HAS_CUDA = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else None
DEVICE = "cuda" if HAS_CUDA else "cpu"


def tier_of(gpu_name):
    """Colab tier from the GPU name: cpu / t4 / l4 / a100 (an unknown GPU is treated like a T4)."""
    if not gpu_name:
        return "cpu"
    name = gpu_name.upper()
    if "A100" in name or "H100" in name:
        return "a100"
    if "L4" in name:
        return "l4"
    return "t4"


TIER = tier_of(GPU_NAME)
if TIER == "cpu" and "VH_NB_SMOKE" not in os.environ:
    SMOKE = True

HF_HOME = os.environ.get("HF_HOME") or ("/content/hf_cache" if IN_COLAB else os.path.expanduser("~/.cache/huggingface"))
os.environ["HF_HOME"] = HF_HOME
os.makedirs(HF_HOME, exist_ok=True)


def version_of(package):
    """Installed version of a distribution without importing it (transformers is pinned in cell 2)."""
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return "-"


ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
disk = shutil.disk_usage("/content" if IN_COLAB else os.getcwd())
print(f"Colab: {IN_COLAB} | GPU: {GPU_NAME or 'none'} -> TIER = {TIER} | DEVICE = {DEVICE} | SMOKE = {SMOKE}")
print(f"RAM {ram_gb:.1f} GB | Disk frei {disk.free / 1e9:.0f} GB | CPUs {os.cpu_count()} | {platform.platform()}")
print(f"python {platform.python_version()} | torch {torch.__version__} | transformers {version_of('transformers')} | "
      f"datasets {version_of('datasets')} | scikit-learn {version_of('scikit-learn')}")
if HAS_CUDA:
    print(f"VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB | bf16 unterstuetzt: {torch.cuda.is_bf16_supported()}")
if IN_COLAB and not os.environ.get("HF_TOKEN"):
    try:                                              # optional Colab secret HF_TOKEN (read token): higher Hub rate limits for parallel sessions
        from google.colab import userdata
        _tok = userdata.get("HF_TOKEN")
        if _tok:
            os.environ["HF_TOKEN"] = _tok
    except Exception:
        pass
print(f"HF_HOME = {HF_HOME} | HF_TOKEN gesetzt: {bool(os.environ.get('HF_TOKEN'))}")

## 2. Installation, Projekt, Drive

Drive mounten, Projekt holen (Default: `git clone` von `GITHUB_URL` auf die lokale Disk; Alternative
`PROJECT_SOURCE = "drive"` = Kopie des Drive-Ordners `ValueHeadsPythia/valueheads_pythia`), gepinnte Abhängigkeiten aus `requirements.txt`
installieren, Ordnerstruktur nach DESIGN 6.4 anlegen. `VH_ROOT` zeigt auf das Drive-Verzeichnis (kompakte Artefakte:
`results/`, `figures/`), `VH_FEATURES` auf `/content/features` (große, flüchtige Feature-Shards); ein begrenzter
Feature-Cache (Cap 4 GB) liegt unter `ValueHeadsPythia/features/`.

In [ ]:
# 2. Installation (gepinnt), Projekt (Drive-Kopie | git clone), Drive mounten, Ordnerstruktur (DESIGN 6.4)
DRIVE_MOUNT = "/content/drive"
DRIVE_ROOT = f"{DRIVE_MOUNT}/MyDrive/ValueHeadsPythia/pythia-1.4b"          # DESIGN 6.4 layout root on Drive
PROJECT_SOURCE = "git"        # "git": clone GITHUB_URL (default) | "drive": project folder on Drive
GITHUB_URL = "https://github.com/Erikiss/ValueHeads.git"  # used with PROJECT_SOURCE == "git"
DRIVE_PROJECT = f"{DRIVE_ROOT}/valueheads_pythia"                # README.md, valueheads_pythia/, scripts/, ... on Drive
LOCAL_PROJECT = "/content/valueheads_pythia"                     # code runs from the local disk


def find_project(start):
    """Project root (the directory holding valueheads_pythia/cli.py) at `start` or one of its parents, else None."""
    d = os.path.abspath(start)
    for _ in range(4):
        if os.path.exists(os.path.join(d, "valueheads_pythia", "cli.py")):
            return d
        d = os.path.dirname(d)
    return None


def pip_install(*spec):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *spec], check=True)


if IN_COLAB:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    if find_project(LOCAL_PROJECT) and PROJECT_SOURCE == "git" and os.path.isdir(os.path.join(LOCAL_PROJECT, ".git")):
        subprocess.run(["git", "-C", LOCAL_PROJECT, "pull", "--ff-only"], check=False)   # pick up fixes without a runtime reset
    if not find_project(LOCAL_PROJECT):
        if PROJECT_SOURCE == "git":
            subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, LOCAL_PROJECT], check=True)
        else:
            # the project root is the directory holding valueheads_pythia/cli.py: MyDrive/ValueHeadsPythia/valueheads_pythia/
            # (documented layout) or MyDrive/ValueHeadsPythia/ itself when the project CONTENTS were copied there
            src = find_project(DRIVE_PROJECT) or find_project(DRIVE_ROOT)
            if not src:
                raise FileNotFoundError(f"Projekt nicht gefunden: Ordner nach {DRIVE_PROJECT} kopieren (Drive) oder "
                                        f"PROJECT_SOURCE = 'git' und GITHUB_URL setzen")
            shutil.copytree(src, LOCAL_PROJECT,
                            ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache", "vh_runs", "features", "results", "figures",
                                                          "hf_cache", "*.npy", "*.npz"))
    PROJECT = LOCAL_PROJECT
    if not os.path.exists(os.path.join(PROJECT, "requirements.txt")):
        raise FileNotFoundError(f"{PROJECT}/requirements.txt fehlt: {PROJECT} ist kein vollstaendiger Projektordner (README.md, "
                                "requirements.txt, valueheads_pythia/, scripts/, data/ erwartet)")
    pip_install("-r", os.path.join(PROJECT, "requirements.txt"))      # torch>=2.1, transformers>=4.56,<6, datasets>=2.19, ...
    if TIER in ("t4", "l4"):
        pip_install("bitsandbytes")                                    # optional: 4-bit judges (ArmoRM, TruthfulQA judges)
    VH_ROOT = DRIVE_ROOT                                               # results/ figures/ on Drive
    VH_FEATURES = "/content/features"                                  # features on the local disk (volatile)
    DRIVE_FEATURE_CACHE = f"{DRIVE_ROOT}/features"                     # capped feature cache on Drive
else:
    PROJECT = os.environ.get("VH_NB_PROJECT") or find_project(os.getcwd()) or find_project(os.path.join(os.getcwd(), ".."))
    if not PROJECT:
        raise FileNotFoundError("Projekt nicht gefunden: VH_NB_PROJECT setzen oder das Notebook aus dem Projekt starten")
    VH_ROOT = os.path.abspath(os.environ.get("VH_ROOT") or os.path.join(PROJECT, "vh_runs"))
    VH_FEATURES = os.path.abspath(os.environ.get("VH_FEATURES") or os.path.join(VH_ROOT, "features"))
    DRIVE_FEATURE_CACHE = os.path.join(VH_ROOT, "features_cache")      # stands in for the Drive cache outside Colab

os.environ.update(VH_ROOT=VH_ROOT, VH_FEATURES=VH_FEATURES)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from valueheads_pythia import config

P = config.paths(VH_ROOT)
for d in [P["heads"], P["directions"], P["eval"], P["judge"], P["sweep"], P["heatmaps"], P["figures_sweep"],
          P["figures_causal"], VH_FEATURES, DRIVE_FEATURE_CACHE]:
    os.makedirs(d, exist_ok=True)
print(f"PROJECT      {PROJECT}\nVH_ROOT      {VH_ROOT}  (results/, figures/)\nVH_FEATURES  {VH_FEATURES}\n"
      f"Feature-Cache {DRIVE_FEATURE_CACHE} (Cap 4 GB)\nHF_HOME      {HF_HOME}")
from valueheads_pythia.cli import COMMANDS

print("CLI-Befehle:", ", ".join(COMMANDS))

## 3. Konfiguration

`MODEL`, `REVISIONS` (Default: die 12 Checkpoints aus DESIGN 3.4), `VALUES`, `MODE` (`pythia` = Anpassungen aus DESIGN 3-5,
`paper` = Repo-Definitionen), `max_samples` aus der Tier-Tabelle (DESIGN 6.3) und dtype/attn_impl aus der Numerik-Policy
(DESIGN 3.1: fp32 für 160m/410m auf jedem Gerät, fp16+sdpa ab 1b auf GPU, nie eager in halber Präzision). `run(...)`
startet einen CLI-Befehl des Projekts in einem eigenen Prozess (Modell-Speicher wird danach frei).

In [ ]:
# 3. Konfiguration: MODEL, REVISIONS, VALUES, MODE, max_samples (Tier), Numerik-Policy, CLI-Runner
MODEL = "pythia-1.4b"                 # pythia-160m | pythia-410m | pythia-1b | pythia-1.4b | pythia-2.8b
MODE = "pythia"                       # "pythia" (DESIGN 3-5) | "paper" (repo definitions, DESIGN 2)
REVISIONS = list(config.CHECKPOINTS_DEFAULT)                  # 12 checkpoints, log-dense early (DESIGN 3.4)
VALUES = ["honesty", "harmless", "helpful"]   # lite run: values with a usable signal
CAUSAL_REVISIONS = list(config.CAUSAL_CHECKPOINTS_DEFAULT)    # likelihood evals: end checkpoint + 2 early (DESIGN 6.2)
SPLIT_SEEDS = [1, 2, 3, 4]            # extra TruthfulQA question splits for ranking stability (DESIGN 5.6)
if SMOKE:
    REVISIONS, VALUES, CAUSAL_REVISIONS, SPLIT_SEEDS = ["step143000"], ["honesty"], ["step143000"], []
CAUSAL_REVISIONS = []                 # lite run: no likelihood evaluation
FINAL_REVISION = REVISIONS[-1]
# Cells 5-10 work on ONE checkpoint (README step 4); the loop over all REVISIONS is the sweep of cell 11 (which cleans the HF
# cache and the volatile features per revision). Set NB_REVISIONS = list(REVISIONS) to run cells 5-10 over every checkpoint
# instead (disk on /content: 160m ~1.4 GB per checkpoint, 1.4b ~8 GB, 2.8b ~14 GB -- cache + features, nothing deleted).
NB_REVISIONS = [FINAL_REVISION]
NB_CAUSAL_REVISIONS = [r for r in CAUSAL_REVISIONS if r in NB_REVISIONS] or [FINAL_REVISION]

EXTRACT_TIER = "smoke" if (SMOKE or TIER == "cpu") else TIER            # max_samples table of DESIGN 6.3
MAX_SAMPLES = {"train": 200, "val": 100} if SMOKE else {}               # smoke: override even the smoke tier (CPU budget)
B = 100 if SMOKE else 1000                                              # cluster-bootstrap resamples (DESIGN 5.3; SMOKE: 100 for the CPU budget)
PERMS = 5 if SMOKE else 100                                             # permutation null (DESIGN 5.4 smoke value is 20 = run_smoke.sh; SMOKE here: 5)
R = 1 if SMOKE else (50 if TIER == "a100" else 20)                      # layer-matched random head sets (DESIGN 5.9)
ALPHAS = [0.5] if SMOKE else list(config.STEER_ALPHAS)                 # steering strengths
RATIO = config.TOP_RATIO                                                # top-15 % rule
STATS_DEVICE = "cuda" if HAS_CUDA else "cpu"                            # permutation refits (batched torch logistic regression)
DTYPE, ATTN_IMPL = config.dtype_policy(MODEL, DEVICE)
PROGRESS = [] if (IN_COLAB and not SMOKE) else ["--no_progress"]
COMMON = ["--model", MODEL, "--mode", MODE]
# Likelihood-evaluation suite per tier (DESIGN 6.2: reduced on T4, full on A100; smoke = one metric, tiny)
EVAL_SUITE = {
    "smoke": {"tqa_pairwise": ["--max_items", "8"]},
    "t4": {"tqa_pairwise": [], "wikitext_ppl": ["--max_tokens", "50000"], "hh_pairwise": [], "helpsteer2_pairwise": [], "tqa_mc1_mc2": []},
    "l4": {"tqa_pairwise": [], "wikitext_ppl": ["--max_tokens", "50000"], "hh_pairwise": [], "helpsteer2_pairwise": [], "tqa_mc1_mc2": []},
    "a100": {"tqa_pairwise": [], "wikitext_ppl": [], "hh_pairwise": [], "helpsteer2_pairwise": [], "tqa_mc1_mc2": []},
}[EXTRACT_TIER]
BATCH_TOKENS = "4096" if DEVICE == "cpu" else "8192"

print(f"MODEL {MODEL} ({config.MODELS[MODEL]['hf_id']}, L/H/D = {config.MODELS[MODEL]['layers']}/{config.MODELS[MODEL]['heads']}/"
      f"{config.MODELS[MODEL]['head_dim']}, k15 = {config.MODELS[MODEL]['k15']}) | MODE {MODE} | TIER {TIER} -> extract tier {EXTRACT_TIER}")
print(f"REVISIONS {REVISIONS}\nVALUES {VALUES}\nCAUSAL_REVISIONS {CAUSAL_REVISIONS} | split seeds {SPLIT_SEEDS}")
print(f"cells 5-10 run on NB_REVISIONS {NB_REVISIONS} (causal: {NB_CAUSAL_REVISIONS}); cell 11 sweeps all REVISIONS")
print(f"B = {B}, P = {PERMS}, R = {R}, alphas = {ALPHAS}, ratio = {RATIO}, stats on {STATS_DEVICE}")
print(f"Numerik-Policy (DESIGN 3.1): dtype = {DTYPE}, attn_implementation = {ATTN_IMPL} on {DEVICE}; forbidden: {config.DTYPE_POLICY['forbidden']}")
print("max_samples (train / val) of tier", EXTRACT_TIER, "for", MODEL, "(None = all):")
for ds in sorted({config.probing_dataset_of(v, MODE) for v in VALUES}):
    ms = [config.max_samples(EXTRACT_TIER, ds, s, MODEL) for s in ("train", "val")]
    print(f"  {ds:12s} {MAX_SAMPLES.get('train', ms[0])} / {MAX_SAMPLES.get('val', ms[1])}" + ("  (SMOKE override)" if MAX_SAMPLES else ""))
print("eval suite:", {m: " ".join(a) or "full" for m, a in EVAL_SUITE.items()})

QUIET_PATTERNS = ("Warning: You are sending", "Loading weights", "UserWarning", "warnings.warn", "FutureWarning")


def run(*args, check=True, extra_env=None):
    """Runs `python -m valueheads_pythia.cli <args>` in a subprocess (fresh process per step: the model's memory is
    released afterwards), streams its output into the notebook and raises on a non-zero exit code."""
    cmd = [sys.executable, "-m", "valueheads_pythia.cli", *map(str, args)]
    env = dict(os.environ, HF_HOME=HF_HOME, VH_ROOT=VH_ROOT, VH_FEATURES=VH_FEATURES, PYTHONUNBUFFERED="1",
               OMP_NUM_THREADS=str(os.cpu_count() or 1), TOKENIZERS_PARALLELISM="false", **(extra_env or {}))
    t0 = time.time()
    print("$ " + " ".join(cmd[2:]), flush=True)
    proc = subprocess.Popen(cmd, cwd=PROJECT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            if not any(p in line for p in QUIET_PATTERNS):
                print(line, end="", flush=True)
        rc = proc.wait()
    except BaseException:
        # cell interrupt: stop the step instead of leaving it running behind the notebook (a second run of the same
        # cell would otherwise write into the same feature / result directories at the same time)
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
        print(f"[abgebrochen nach {time.time() - t0:.0f}s: {' '.join(cmd[2:4])}]", flush=True)
        raise
    print(f"[exit {rc}, {time.time() - t0:.0f}s]", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"command failed (exit {rc}): {' '.join(cmd[2:])}")
    return rc


run("--help")

## 4. Selftest

Hook-Layout (V-Zeilen von Head 7 in Layer 5 nullen → nur Slice 7 wird 0), Masken-Positionen (`positions="all"` ändert
log p an allen Positionen, `"last"` nur an der letzten, decode-only-Hooks wirken erst im KV-Cache-Decode) und
Datenzählungen (653/164 Fragen, 4 712/1 206 Texte, HelpSteer2-Zählungen, HH-Paare). Bricht bei Abweichung ab.

In [ ]:
# 4. Selftest: Hook-Layout + Masken-Positionen + Datenzaehlungen (exit 1 bei Abweichung -> Notebook stoppt)
run("selftest", "--model", MODEL, "--revision", FINAL_REVISION, "--device", DEVICE, "--dtype", DTYPE, "--attn_impl", ATTN_IMPL,
    "--data", "none" if SMOKE else "quick")

## 5. Extraktion

Schleife über Datensatz × Split für `NB_REVISIONS` (Default: nur `FINAL_REVISION`; alle Checkpoints extrahiert der Sweep in
Zelle 11, HelpSteer2-Werte teilen den Extract-once-Satz `helpsteer2`). Jede
Extraktion ist resumable (Layer-Shards + `DONE`-Marker): nach einem Session-Abbruch einfach die Zelle erneut starten.
Ein begrenzter Feature-Cache auf Drive (honesty train+val, alle Val-Splits; Cap 4 GB) wird vor der Extraktion
zurückgespielt und danach aufgefüllt.

In [ ]:
# 5. Extraktion (Datensatz x Split je Checkpoint, DONE-Marker, Resume, Drive-Feature-Cache)
from valueheads_pythia.extract import is_done, read_manifest

CACHE_CAP_GB = 1.0


def feature_triples(revisions, values):
    """(revision, dataset, split) triples to extract; HelpSteer2 values share one extract-once set in mode pythia."""
    seen, out = set(), []
    for rev in revisions:
        for v in values:
            ds = config.probing_dataset_of(v, MODE)
            for split in ("train", "val"):
                if (rev, ds, split) not in seen:
                    seen.add((rev, ds, split))
                    out.append((rev, ds, split))
    return out


def dir_size(path):
    """Total size of the files below `path` in bytes."""
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) if os.path.isdir(path) else 0


def cacheable(dataset, split):
    """DESIGN 6.4: the Drive cache keeps honesty (train + val) and every val split."""
    return dataset == "honesty" or split == "val"


def cache_dir_of(rev, dataset, split):
    return os.path.join(DRIVE_FEATURE_CACHE, MODEL, rev, f"{dataset}_{split}")


def cache_pull(rev, dataset, split):
    """Restores a finished feature directory from the Drive cache when the local copy is missing."""
    local, cached = config.feature_dir(P, MODEL, rev, dataset, split), cache_dir_of(rev, dataset, split)
    if is_done(cached) and not is_done(local):
        shutil.copytree(cached, local, dirs_exist_ok=True)
        print(f"[cache] restored {local} from {cached}")
        return True
    return False


def cache_push(rev, dataset, split):
    """Copies a finished, cacheable feature directory to the Drive cache while the cache stays below the cap."""
    local, cached = config.feature_dir(P, MODEL, rev, dataset, split), cache_dir_of(rev, dataset, split)
    if not (cacheable(dataset, split) and is_done(local)) or is_done(cached):
        return False
    if (dir_size(DRIVE_FEATURE_CACHE) + dir_size(local)) / 1e9 > CACHE_CAP_GB:
        print(f"[cache] cap {CACHE_CAP_GB} GB reached, not caching {dataset}_{split}@{rev}")
        return False
    shutil.copytree(local, cached, dirs_exist_ok=True)
    print(f"[cache] stored {cached} ({dir_size(cached) / 1e6:.1f} MB)")
    return True


t_extract = time.time()
for rev, ds, split in feature_triples(NB_REVISIONS, VALUES):       # one checkpoint (cell 3); the sweep (cell 11) does the rest
    cache_pull(rev, ds, split)
    args = ["extract", *COMMON, "--revision", rev, "--dataset", ds, "--split", split, "--tier", EXTRACT_TIER,
            "--device", DEVICE, "--dtype", DTYPE, "--attn_impl", ATTN_IMPL, *PROGRESS]
    if MAX_SAMPLES:
        args += ["--max_samples", MAX_SAMPLES[split]]
    run(*args)
    cache_push(rev, ds, split)
    man = read_manifest(config.feature_dir(P, MODEL, rev, ds, split))
    print(f"    -> n={man['n']} tokens={man['n_tokens']} dtype={man.get('dtype')}/{man.get('attn_impl')} "
          f"{man.get('tokens_per_second')} tok/s")
print(f"[extract] all features present after {time.time() - t_extract:.0f}s; cache {dir_size(DRIVE_FEATURE_CACHE) / 1e9:.2f} GB")

## 6. Probing + Statistik + Kontrollen

Eine logistische Regression je Head (`probe`, Features head-only = Primär und [Head ⊕ Layer-Mittel] = Repo-Vergleich),
dann `probe-stats`: Cluster-Bootstrap-CIs, Permutations-Null für das Max-über-Heads (Westfall-Young), signifikante
Menge und Selektions-/Konfirmations-Split (DESIGN 5.3-5.5). Kontrollen nach DESIGN 5.7 für den Endcheckpoint:
Oberflächen-Probe, Last-Token-Embedding-Probe, Layer-Mittel-Probe – ein Head gilt nur als wert-sensitiv, wenn seine AUC
die beste Kontrolle im gepaarten Cluster-Bootstrap übertrifft (`heads_vs_controls` auf den gecachten Val-Scores,
CI-Untergrenze von AUC(Head) − AUC(Kontrolle) > 0). HelpSteer2-Werte bekommen zusätzlich die gruppierte 5-fach-CV-Tabelle
über train + val (`probe --cv 5`, DESIGN 5.3). Die Ergebnis-JSONs liegen unter `VH_ROOT/results/heads/` (Drive).

In [ ]:
# 6. Probing (head-only + head+mean) + Statistik (Bootstrap, Permutations-Null, Konfirmations-Split) + Kontrollen
import gc

import numpy as np
import pandas as pd

from valueheads_pythia.analysis.sweep_curves import summarize_table
from valueheads_pythia.commands.probe import resolve_feature_dirs
from valueheads_pythia.controls import best_control, embedding_probe, heads_vs_controls, layer_mean_probe, surface_probe
from valueheads_pythia.data import extract_texts, texts_fingerprint
from valueheads_pythia.probing import load_head_scores
from valueheads_pythia.probing_stats import load_valscores, valscores_path

for rev in NB_REVISIONS:
    for v in VALUES:
        common = [*COMMON, "--revision", rev, "--value", v]
        run("probe", *common, "--features", "both", "--quiet")
        run("probe-stats", *common, "--features", "both", "--bootstrap", B, "--perms", PERMS, "--device", STATS_DEVICE, "--quiet")
        if v in config.HELPSTEER_VALUES_EXT:
            # DESIGN 5.3: HelpSteer2 values are additionally ranked by a grouped 5-fold CV over train + val (group = prompt)
            run("probe", *common, "--cv", 5, "--quiet")


def labeled_texts(dataset, split, value, manifest):
    """Texts + labels of a feature directory, re-derived from its manifest (same subsample, same seeds)."""
    texts, labels, groups = extract_texts(dataset, split, MODE, manifest.get("max_samples"), int(manifest.get("seed", 42)),
                                          int(manifest.get("split_seed", 42)))
    if manifest.get("texts_sha256") and texts_fingerprint(texts) != manifest["texts_sha256"]:
        raise ValueError(f"re-derived {dataset}/{split} texts differ from the extracted ones (manifest {manifest.get('max_samples')}/{manifest.get('seed')})")
    if isinstance(labels, dict):
        labels = labels[value]
    labels = np.asarray(labels)
    keep = labels >= 0
    return [t for t, k in zip(texts, keep) if k], labels[keep], (None if groups is None else np.asarray(groups)[keep])


def run_controls(rev, value):
    """DESIGN 5.7 controls (ii)-(iv) for one (revision, value): AUCs of the surface, embedding and layer-mean probes
    next to the top head; written to results/controls_{model}_{rev}_{value}.json."""
    out_path = os.path.join(P["results"], f"controls_{MODEL}_{rev}_{value}.json")
    if os.path.exists(out_path):
        with open(out_path) as f:
            return json.load(f)
    train_dir, val_dir, label_value = resolve_feature_dirs(P, MODEL, rev, value, MODE)
    ds = config.probing_dataset_of(value, MODE)
    tr_texts, tr_y, tr_g = labeled_texts(ds, "train", value, read_manifest(train_dir))
    va_texts, va_y, _ = labeled_texts(ds, "val", value, read_manifest(val_dir))
    surface = surface_probe(tr_texts, tr_y, tr_g, va_texts, va_y, return_scores=True)
    from transformers.utils import logging as hf_logging
    from valueheads_pythia.models import load_model
    hf_logging.set_verbosity_error()
    hf_logging.disable_progress_bar()
    model, tokenizer = load_model(MODEL, revision=rev, device=DEVICE, dtype=DTYPE, attn_impl=ATTN_IMPL)
    embedding = embedding_probe(tr_texts, tr_y, va_texts, va_y, model=model, tokenizer=tokenizer, return_scores=True)
    del model
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()
    layer_mean = layer_mean_probe(train_dir, val_dir, value=label_value, return_scores=True)
    best = best_control(surface, embedding, layer_mean)
    heads_json = config.heads_path(P, MODEL, rev, value, MODE)
    df, _ = load_head_scores(heads_json)
    top = summarize_table(df, RATIO)
    # DESIGN 5.7: paired cluster bootstrap of AUC(head) - AUC(best control) on the cached val decision scores of probe-stats
    head_scores, labels, groups, _ = load_valscores(valscores_path(heads_json))
    if not np.array_equal(labels, va_y):
        raise ValueError("val labels of the cached head scores differ from the re-derived val set")
    paired = heads_vs_controls(head_scores, labels, groups, {"surface": surface["scores"], "embedding": embedding["scores"],
                                                            "layer_mean": layer_mean.attrs["scores"][best["best_layer"]]}, B=B, seed=0)
    res = {"model": MODEL, "revision": rev, "value": value, "surface_auc": surface["auc"], "surface_coef": surface["coef"],
           "embedding_auc": embedding["auc"], "layer_mean_auc_by_layer": layer_mean["auc"].round(4).tolist(), **best,
           "top_head": f"L{top['top_layer']}.H{top['top_head']}", "top_auc": top["top_auc"], "top_ci": [top["ci_lo"], top["ci_hi"]],
           "perm_thr": top["perm_thr"], "n_signif": top["n_signif"], "conf_top1": top["conf_top1"],
           "n_heads_above_best_control": int((df["auc"] > best["best"]).sum()),
           "paired_best_control": paired["best_name"], "n_heads_beating_best_control": paired["n_beats"],
           "heads_beating_best_control": paired["beating_heads"], "n_beats_by_control": paired["n_beats_by_control"],
           "paired_bootstrap_B": B}
    with open(out_path, "w") as f:
        json.dump(res, f, indent=1)
    return res


rows = []
for v in VALUES:
    c = run_controls(FINAL_REVISION, v)
    rows.append({k: c[k] for k in ["value", "top_head", "top_auc", "perm_thr", "n_signif", "conf_top1", "surface_auc", "embedding_auc",
                                   "layer_mean", "best_name", "n_heads_above_best_control", "n_heads_beating_best_control"]})
print(f"Kontrollen @ {FINAL_REVISION} (AUC auf Val; Head wert-sensitiv nur, wenn er die beste Kontrolle im gepaarten "
      f"Cluster-Bootstrap uebertrifft: n_heads_beating_best_control, B = {B}):")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## 7. Heatmaps + Layer-Profile

Fig.-2-Stil je Wert: head-only-AUC (Primärscore, signifikante Heads als Punkte, Top-15 % als Quadrate) und daneben der
Repo-Score (normierte quadrierte Accuracy des [Head ⊕ Layer-Mittel]-Probes, vmin 0.6). Dazu die Layer-Profile
(max/mean AUC, Anzahl Top-k- und signifikante Heads je Layer).

In [ ]:
# 7. Heatmaps (head-only AUC und Repo-Score) + Layer-Profile ueber die Python-API
from IPython.display import Image, display

from valueheads_pythia.analysis.heatmaps import heatmap_figure, layer_profile, layer_profile_figure


def head_tables(rev, feature_tag=None):
    """{value: head table} of the probed values at `rev` (feature_tag None = primary, 'headmean' = repo features)."""
    out = {}
    for v in VALUES:
        path = config.heads_path(P, MODEL, rev, v, MODE, feature_tag)
        if os.path.exists(path):
            out[v] = load_head_scores(path)[0]
    return out


for rev in NB_REVISIONS:
    primary = head_tables(rev)
    repo = head_tables(rev, "headmean") if MODE == "pythia" else {}     # mode paper: the primary table already uses the repo features
    if not primary:
        continue
    tag = "" if MODE == "pythia" else f"_{MODE}"
    fig_primary = os.path.join(P["heatmaps"], f"{MODEL}_{rev}{tag}_nb_auc.png")
    heatmap_figure(primary, fig_primary, mode=MODE, ratio=RATIO, title=f"{MODEL} @ {rev}: head-only {'AUC' if MODE == 'pythia' else 'Repo-Score'}")
    fig_layers = os.path.join(P["heatmaps"], f"{MODEL}_{rev}{tag}_nb_layers.png")
    layer_profile_figure(primary, fig_layers, score="auc", ratio=RATIO, title=f"{MODEL} @ {rev}: Layer-Profile")
    figs = [fig_primary, fig_layers]
    if repo:
        fig_repo = os.path.join(P["heatmaps"], f"{MODEL}_{rev}{tag}_nb_reposcore.png")
        heatmap_figure(repo, fig_repo, mode="paper", ratio=RATIO, title=f"{MODEL} @ {rev}: Repo-Score ([Head ; Layer-Mittel], normierte Accuracy^2)")
        figs.append(fig_repo)
    for v, df in primary.items():
        prof = layer_profile(df, "auc", RATIO)
        print(f"{rev} {v}: Layer mit max AUC = {int(prof['auc_max'].idxmax())} ({prof['auc_max'].max():.3f}); "
              f"Top-k je Layer {prof['n_topk'].tolist()}; signifikant je Layer {prof['n_signif'].tolist()}")
    if rev == FINAL_REVISION:
        for f in figs:
            display(Image(filename=f, width=900))
print(f"Heatmaps unter {P['heatmaps']}")

## 8. Richtungen + σ

Mass-Mean-Richtung je (Layer, Head) (Eq. 2, Einheitsnorm) und σ = Standardabweichung der projizierten
Val-Aktivierungen des STD-Sets (im `pythia`-Modus aus Train/Val-disjunkten Texten). Ergebnis
`results/directions/{model}_{revision}_{value}.npz`.

In [ ]:
# 8. Richtungen (Mass-Mean, Einheitsnorm) + sigma je STD-Set
from valueheads_pythia.directions import load_directions

for rev in NB_REVISIONS:
    for v in VALUES:
        run("directions", *COMMON, "--revision", rev, "--value", v)
for v in VALUES:
    directions, sigmas, meta = load_directions(config.directions_path(P, MODEL, FINAL_REVISION, v, MODE))
    norms = np.linalg.norm(directions, axis=1)       # raw mass-mean norms; unit-normalized when the steering vectors are built
    print(f"{v}@{FINAL_REVISION}: directions {directions.shape}, |dir| median={np.median(norms):.3f} max={norms.max():.3f}, "
          + ", ".join(f"sigma[{s}] median={np.median(x):.3f}" for s, x in sigmas.items()) + f", from {meta.get('feature_dirs')}")

## 11. Sweep-Treiber über `REVISIONS`

`sweep` ruft je Checkpoint die Schritte 5-8 auf (extract → probe → probe-stats → directions; Likelihood-Eval nur für
`CAUSAL_REVISIONS`), überspringt erledigte Tripel (idempotent, nach Session-Abbruch einfach erneut starten), schreibt
`results/manifest.json` (dtype, attn_impl, Versionen, Commit-SHAs) und `results/sweep/*_curve.csv`. Auf Colab wird jede
Revision nach Gebrauch aus dem HF-Cache gelöscht (`--cleanup_cache`; 2.8b: 5.7 GB je Revision) und ihre nicht cachebaren
Train-Features werden von `/content/features` entfernt (`--cleanup_features`, DESIGN 6.4; 1.4b: 5.2 GB, 2.8b: 8.7 GB je Checkpoint);
`FINAL_REVISION` bleibt erhalten. Werte, die laut Manifest fertig sind, werden beim Neustart nicht erneut extrahiert.

In [ ]:
# 11. Sweep-Treiber ueber REVISIONS (ruft 5-9 je Checkpoint; Cache-Cleanup je Revision; idempotent)
sweep = ["sweep", *COMMON, "--revisions", *REVISIONS, "--values", *VALUES, "--tier", EXTRACT_TIER, "--features", "both",
         "--bootstrap", B, "--perms", PERMS, "--stats_device", STATS_DEVICE, "--device", DEVICE, "--dtype", DTYPE, "--attn_impl", ATTN_IMPL,
         "--eval_revisions", *CAUSAL_REVISIONS,
         "--eval_args", " ".join(["--metric", "tqa_pairwise", "--intervention", "mask", "--select", "topk", "--random_sets", str(R),
                                  "--positions", "all", *EVAL_SUITE.get("tqa_pairwise", []), *PROGRESS]),
         "--continue_on_error", *PROGRESS]
if MAX_SAMPLES:
    sweep += ["--max_samples_train", MAX_SAMPLES["train"], "--max_samples_val", MAX_SAMPLES["val"]]
if SPLIT_SEEDS:
    sweep += ["--split_seeds", *SPLIT_SEEDS]
if IN_COLAB:
    # per revision: drop the HF snapshot and the non-cacheable local train features (harmless / helpful / helpsteer2;
    # honesty + val splits stay for the Drive cache); FINAL_REVISION keeps both (cells 5-10 work on it)
    sweep += ["--cleanup_cache", "--cleanup_features", "--keep_revisions", "main", FINAL_REVISION]
run(*sweep, "--dry_run")
run(*sweep)
for rev, ds, split in feature_triples(REVISIONS, VALUES):      # top up the Drive feature cache (val splits, honesty)
    cache_push(rev, ds, split)
with open(P["manifest"]) as f:
    manifest = json.load(f)
print(f"manifest: tier {manifest.get('tier')}, revisions {list(manifest.get('revisions', {}))}; "
      + "; ".join(f"{r}: {e.get('dtype')}/{e.get('attn_impl')} sha {str(e.get('commit_sha'))[:8]}" for r, e in manifest.get("revisions", {}).items()))

## 12. Analyse, Drive-Sync, Ausgaben löschen

`analyze`: Sweep-Kurven (Top-AUC, n_signif, Spearman/Jaccard zu step143000 über log step), Korrelationen über Checkpoints
und Werte, Richtungswinkel, `summary_{model}.csv` und der deutsche Report `results/report_{model}.md`. Danach Sync des
Feature-Caches nach Drive und Größenübersicht. **Vor dem Speichern: Bearbeiten → Alle Ausgaben löschen** (HH-RLHF-,
BeaverTails- und Jailbreak-Inhalte dürfen nie in gespeicherten Ausgaben stehen).

In [ ]:
# 12. Analyse (Sweep-Kurven, Korrelationen, Winkel, Report) + Drive-Sync + Erinnerung "Alle Ausgaben loeschen"
run("analyze", "--model", MODEL, "--ratio", RATIO, "--final_revision", FINAL_REVISION)

for v in VALUES:
    tag = "" if MODE == "pythia" else f"_{MODE}"
    curve = os.path.join(P["sweep"], f"{MODEL}_{v}{tag}_curve.csv")
    fig = os.path.join(P["figures_sweep"], f"{MODEL}_{v}{tag}.png")
    if os.path.exists(curve):
        print(pd.read_csv(curve)[["revision", "top_layer", "top_head", "top_auc", "ci_lo", "ci_hi", "perm_thr", "n_signif",
                                   "spearman_to_final", "jaccard_to_final"]].to_string(index=False, float_format=lambda x: f"{x:.3f}"))
        if os.path.exists(fig):
            display(Image(filename=fig, width=900))
    else:
        print(f"{v}: Sweep-Kurve braucht >= 2 Checkpoints (vorhanden: {len(REVISIONS)})")
report = os.path.join(P["results"], f"report_{MODEL}.md")
with open(report) as f:
    lines = f.read().splitlines()
print("\n".join(lines[:40]) + ("\n[...]" if len(lines) > 40 else ""))

# Drive sync: VH_ROOT already lives on Drive (results/, figures/); the feature cache is topped up and flushed
for rev, ds, split in feature_triples(REVISIONS, VALUES):
    cache_push(rev, ds, split)
os.sync()
print("\nGroessen:", ", ".join(f"{name} {dir_size(path) / 1e9:.2f} GB" for name, path in
                               [("results", P["results"]), ("figures", P["figures"]), ("features (lokal)", VH_FEATURES),
                                ("feature cache", DRIVE_FEATURE_CACHE)]))
if IN_COLAB:
    print(f"Artefakte auf Drive: {VH_ROOT}/results, {VH_ROOT}/figures; Report {report}")
print("\n*** Vor dem Speichern des Notebooks: Bearbeiten -> Alle Ausgaben loeschen (Edit -> Clear all outputs). ***")

---
**Erinnerung:** *Bearbeiten → Alle Ausgaben löschen* (Edit → Clear all outputs), bevor das Notebook gespeichert, auf
Drive abgelegt oder geteilt wird. Das Repository hält das Notebook ohne Ausgaben (`nbstripout`).